# ANOVA em Python: companion das notas de aula

Este notebook acompanha as notas `GCC1625_05_anova.pdf`. A teoria, as fórmulas e a motivação estão nas notas; aqui o foco é transformar esses conceitos em código executável.

| Nas notas | Neste notebook |
|---|---|
| Variável independente e variável dependente | Identificação de colunas e separação dos grupos |
| Variabilidade inter-grupos e intra-grupos | Cálculo manual de `SS`, `MS`, graus de liberdade e estatística `F` |
| Condições de aplicabilidade | Levene, Shapiro-Wilk e visualização por grupo |
| ANOVA de uma via | `scipy.stats.f_oneway` |
| Teste post-hoc de Tukey | `statsmodels.stats.multicomp.pairwise_tukeyhsd` |

Em todos os exemplos, use `alpha = 0.05`.


In [ ]:
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import f_oneway, levene, shapiro
from statsmodels.stats.multicomp import pairwise_tukeyhsd

ALPHA = 0.05

# Funciona tanto quando o notebook é executado a partir de notebooks/ quanto da raiz do repositório.
DATA_DIR = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"
DATA_DIR


## Funções auxiliares

As funções abaixo encapsulam o roteiro operacional da aula: separar grupos, resumir os dados, verificar condições, aplicar ANOVA e, quando necessário, aplicar Tukey.


In [ ]:
def separar_grupos(df, grupo_col, valor_col):
    """Retorna um dicionário {nome_do_grupo: valores} sem valores ausentes."""
    return {
        grupo: dados[valor_col].dropna().to_numpy()
        for grupo, dados in df.groupby(grupo_col, sort=True)
    }


def resumo_por_grupo(df, grupo_col, valor_col):
    return (
        df.groupby(grupo_col)[valor_col]
        .agg(n="count", media="mean", desvio_padrao="std", variancia="var")
        .round(4)
    )


def verificar_condicoes(grupos, alpha=ALPHA):
    """Aplica Levene para homocedasticidade e Shapiro-Wilk para normalidade por grupo."""
    lev_stat, lev_p = levene(*grupos.values(), center="mean")
    linhas = [{
        "verificacao": "Levene (variancias iguais)",
        "grupo": "todos",
        "estatistica": lev_stat,
        "p_valor": lev_p,
        "nao_rejeita_H0": lev_p > alpha,
    }]

    for nome, valores in grupos.items():
        shap_stat, shap_p = shapiro(valores)
        linhas.append({
            "verificacao": "Shapiro-Wilk (normalidade)",
            "grupo": nome,
            "estatistica": shap_stat,
            "p_valor": shap_p,
            "nao_rejeita_H0": shap_p > alpha,
        })

    return pd.DataFrame(linhas).round(4)


def aplicar_anova(grupos):
    f_stat, p_value = f_oneway(*grupos.values())
    return pd.Series({"F": f_stat, "p_valor": p_value, "rejeita_H0": p_value <= ALPHA}).round(4)


def aplicar_tukey(df, grupo_col, valor_col, alpha=ALPHA):
    tukey = pairwise_tukeyhsd(endog=df[valor_col], groups=df[grupo_col], alpha=alpha)
    tabela = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
    return tukey, tabela


def plotar_tukey(tukey_df, titulo):
    dados = tukey_df.copy()
    dados["comparacao"] = dados["group1"].astype(str) + " vs " + dados["group2"].astype(str)
    dados = dados.sort_values("meandiff").reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    y = np.arange(len(dados))
    ax.hlines(y=y, xmin=dados["lower"], xmax=dados["upper"], color="black")
    ax.plot(dados["meandiff"], y, "o", color="tab:red")
    ax.axvline(0, color="tab:blue", linestyle="--", label="diferença = 0")
    ax.set_yticks(y)
    ax.set_yticklabels(dados["comparacao"])
    ax.set_xlabel("Diferença de médias")
    ax.set_title(titulo)
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    return ax


## Exemplo 1: PlantGrowth

Este exemplo corresponde à Seção 5.1 das notas. A variável independente é `group` (`ctrl`, `trt1`, `trt2`) e a variável dependente é `weight`.


In [ ]:
try:
    frutos_df = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/datasets/PlantGrowth.csv")
except Exception:
    frutos_df = pd.read_csv(StringIO('rownames,weight,group\n1,4.17,ctrl\n2,5.58,ctrl\n3,5.18,ctrl\n4,6.11,ctrl\n5,4.50,ctrl\n6,4.61,ctrl\n7,5.17,ctrl\n8,4.53,ctrl\n9,5.33,ctrl\n10,5.14,ctrl\n11,4.81,trt1\n12,4.17,trt1\n13,4.41,trt1\n14,3.59,trt1\n15,5.87,trt1\n16,3.83,trt1\n17,6.03,trt1\n18,4.89,trt1\n19,4.32,trt1\n20,4.69,trt1\n21,6.31,trt2\n22,5.12,trt2\n23,5.54,trt2\n24,5.50,trt2\n25,5.37,trt2\n26,5.29,trt2\n27,4.92,trt2\n28,6.15,trt2\n29,5.80,trt2\n30,5.26,trt2\n'))

frutos_df.head()


In [ ]:
resumo_por_grupo(frutos_df, "group", "weight")


In [ ]:
ax = frutos_df.boxplot(column="weight", by="group", grid=False)
ax.set_title("Peso por grupo de tratamento")
ax.set_xlabel("Grupo")
ax.set_ylabel("Peso")
plt.suptitle("")
plt.show()


### Verificação das condições

O teste de Levene avalia a hipótese nula de variâncias iguais. Para normalidade, usamos Shapiro-Wilk porque cada grupo tem apenas 10 observações; com amostras pequenas, testes de normalidade devem ser interpretados com cautela.


In [ ]:
grupos_frutos = separar_grupos(frutos_df, "group", "weight")
verificar_condicoes(grupos_frutos)


### ANOVA manual: da fórmula ao código

A célula abaixo implementa a decomposição apresentada nas notas: soma de quadrados entre grupos, soma de quadrados dentro dos grupos, graus de liberdade, quadrados médios e estatística `F`.


In [ ]:
def tabela_anova_manual(grupos):
    valores = np.concatenate(list(grupos.values()))
    media_global = valores.mean()
    k = len(grupos)
    n_total = len(valores)

    ss_entre = sum(len(x) * (x.mean() - media_global) ** 2 for x in grupos.values())
    ss_dentro = sum(((x - x.mean()) ** 2).sum() for x in grupos.values())
    ss_total = ((valores - media_global) ** 2).sum()

    gl_entre = k - 1
    gl_dentro = n_total - k
    gl_total = n_total - 1

    ms_entre = ss_entre / gl_entre
    ms_dentro = ss_dentro / gl_dentro
    f_manual = ms_entre / ms_dentro

    return pd.DataFrame({
        "fonte": ["entre grupos", "dentro dos grupos", "total"],
        "SS": [ss_entre, ss_dentro, ss_total],
        "gl": [gl_entre, gl_dentro, gl_total],
        "MS": [ms_entre, ms_dentro, np.nan],
        "F": [f_manual, np.nan, np.nan],
    }).round(4)

anova_manual_frutos = tabela_anova_manual(grupos_frutos)
anova_manual_frutos


In [ ]:
resultado_frutos = aplicar_anova(grupos_frutos)
resultado_frutos


A tabela manual deve produzir a mesma estatística `F` retornada por `f_oneway`. A diferença é que `f_oneway` já encapsula o cálculo e retorna diretamente `F` e o p-valor.

Para `PlantGrowth`, espera-se rejeitar `H0` ao nível de 5%, indicando evidência de que pelo menos um grupo tem média populacional diferente. A ANOVA não diz qual par difere; essa pergunta é de um teste post-hoc.


## Exemplo 2: SandwichAnts

Este exemplo corresponde às Seções 5.2 e 5.3 das notas. A variável independente é `Filling` e a variável dependente é `Ants`.


In [ ]:
sanduiches_df = pd.read_csv(DATA_DIR / "SandwichAnts.csv")
sanduiches_df.head()


In [ ]:
resumo_por_grupo(sanduiches_df, "Filling", "Ants")


In [ ]:
ax = sanduiches_df.boxplot(column="Ants", by="Filling", grid=False, rot=20)
ax.set_title("Formigas atraídas por tipo de recheio")
ax.set_xlabel("Recheio")
ax.set_ylabel("Número de formigas")
plt.suptitle("")
plt.show()


In [ ]:
grupos_formigas = separar_grupos(sanduiches_df, "Filling", "Ants")
verificar_condicoes(grupos_formigas)


In [ ]:
resultado_formigas = aplicar_anova(grupos_formigas)
resultado_formigas


Se a ANOVA rejeitar `H0`, aplicamos Tukey para descobrir quais pares de recheios diferem. A coluna `reject` indica rejeição da igualdade de médias para aquele par, já com ajuste para múltiplas comparações.


In [ ]:
tukey_formigas, tukey_formigas_df = aplicar_tukey(sanduiches_df, "Filling", "Ants")
tukey_formigas_df


In [ ]:
plotar_tukey(tukey_formigas_df, "Teste de Tukey: recheios dos sanduiches")
plt.show()


No gráfico de Tukey, intervalos que não cruzam a linha vertical em zero indicam pares com diferença estatisticamente significativa. Essa visualização complementa a coluna `reject` da tabela.


## Exercícios guiados

Os exercícios abaixo usam os arquivos indicados nas notas. Para cada conjunto, carregue os dados, inspecione os grupos, verifique as condições, aplique ANOVA e use Tukey apenas quando a ANOVA rejeitar `H0`.


In [ ]:
arquivos_exercicios = {
    "metodo_estudo": DATA_DIR / "anova" / "MetodoEstudo.csv",
    "fertilizante": DATA_DIR / "anova" / "Fertilizante.csv",
    "satisfacao_celular": DATA_DIR / "anova" / "SatisfacaoCelular.csv",
}

for nome, arquivo in arquivos_exercicios.items():
    df = pd.read_csv(arquivo)
    print(f"\n{nome}: {df.shape}")
    display(df.head())
    display(resumo_por_grupo(df, "Metodo", "Nota"))


In [ ]:
def roteiro_anova(df, grupo_col="Metodo", valor_col="Nota"):
    grupos = separar_grupos(df, grupo_col, valor_col)
    print("Resumo por grupo")
    display(resumo_por_grupo(df, grupo_col, valor_col))

    print("\nVerificacao das condicoes")
    display(verificar_condicoes(grupos))

    print("\nANOVA")
    resultado = aplicar_anova(grupos)
    display(resultado)

    if bool(resultado["rejeita_H0"]):
        print("\nTukey")
        _, tukey_df = aplicar_tukey(df, grupo_col, valor_col)
        display(tukey_df)
        plotar_tukey(tukey_df, f"Teste de Tukey: {valor_col} por {grupo_col}")
        plt.show()
    else:
        print("\nComo a ANOVA nao rejeitou H0, Tukey nao e necessario.")


### Exercício 1: métodos de estudo


In [ ]:
metodo_df = pd.read_csv(arquivos_exercicios["metodo_estudo"])
roteiro_anova(metodo_df)


### Exercício 2: fertilizantes


In [ ]:
fertilizante_df = pd.read_csv(arquivos_exercicios["fertilizante"])
roteiro_anova(fertilizante_df)


### Exercício 3: satisfação com modelos de celular


In [ ]:
satisfacao_df = pd.read_csv(arquivos_exercicios["satisfacao_celular"])
roteiro_anova(satisfacao_df)


## Fechamento

Use este notebook como checklist computacional para a ANOVA de uma via:

1. Identifique a variável de grupo e a variável numérica.
2. Visualize e resuma os grupos.
3. Verifique homocedasticidade e normalidade.
4. Aplique ANOVA.
5. Se `H0` for rejeitada, aplique Tukey para localizar os pares diferentes.
6. Interprete o resultado no contexto do problema, não apenas pelo p-valor.
